In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
#build vocabulary of characters and mappings to/from int
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [61]:

#set a blocksize i.e. how many characters will be considered at a time
blocksize = 8


######build the dataset WITH SPLITS
def build_dataset(words):

    #create an X and Y mapping arrays
    X, Y = [],[]
    
    #for every word
    
    for w in words:
        #create a context list of length blocksize
        context = [0]*blocksize
    
        for ch in w + '.':
            #get the index of the char
            ix = stoi[ch]
            #add the context that produced this char to X list
            X.append(context)
            
            #add the output (ix) to the Y list
            Y.append(ix)
            
            #crop and append the context
            context = context[1:]+[ix]
        #set X and Y to torch tensors
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

#split the dataset
import random
random.seed(42)
random.shuffle(words)

n1 = int(0.8*len(words))
n2 = int(0.9*len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])



In [65]:
for x,y in zip(Xtr[:20], Ytr[:20]) :
    print(''.join(itos[ix.item()] for ix in x), '--->', itos[y.item()])

........ ---> t
.......t ---> a
......ta ---> u
.....tau ---> r
....taur ---> e
...taure ---> n
..tauren ---> .
........ ---> s
.......s ---> u
......su ---> l
.....sul ---> e
....sule ---> m
...sulem ---> a
..sulema ---> n
.suleman ---> .
........ ---> z
.......z ---> e
......ze ---> r
.....zer ---> e
....zere ---> n


In [121]:
#let's train a deeper network

#class linear (layer)
class Linear:
    
    def __init__(self, fanin, fanout, bias=True):
        #init weights
        self.weight = torch.randn((fanin, fanout)) #/ fanin**0.5 
        #init biases
        self.bias = torch.zeros(fanout) if bias else None

    def __call__(self, x):
        #matmul weights with x
        self.out = x @ self.weight 
        #add bias
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

class BatchNorm1d:

    def __init__(self, dim, eps=1e-5, momentum = 0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        #parameters (trained with backprop)
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        #buffers
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):
        if self.training:
            if x.ndim == 2:
                dim = 0
            if x.ndim == 3:
                dim = (0,1)
            xmean = x.mean(dim, keepdim=True)
            xvar = x.var(dim, keepdim=True)
        else:
            xmean = self.running_mean
            xvar = self.running_var
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps) #normalize unit variance
        self.out = self.gamma * xhat + self.beta 

        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out
        
    def parameters(self):
        return []

class Embed:
    def __init__(self, num_embeddings, embedding_dim):
        self.weight = torch.rand((num_embeddings, embedding_dim))

    def __call__(self, ix):
        self.out = self.weight[ix]
        return self.out

    def parameters(self):
        return [self.weight]

class FlattenConsecutive:
    def __init__(self, n):
        self.n = n
    
    def __call__(self, x):
        B, T, C = x.shape
        n = self.n
        x = x.view(B, T//n, C*n)
        
        if x.shape[1] == 1:
            x=x.squeeze(1)

        self.out = x
        return self.out

    def parameters(self):
        return []

class Sequential:
    def __init__(self, layers):
        self.layers = layers

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        self.out = x
        return self.out

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [122]:
torch.manual_seed(42)

n_embed = 24
n_hidden = 128
vocab_size = len(itos)


model = Sequential([
    Embed(vocab_size, n_embed), 
    FlattenConsecutive(2), Linear(n_embed * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, vocab_size)
])

with torch.no_grad():
    model.layers[-1].weight *= 0.1 #make last layer less confident


parameters = model.parameters()
print(sum(p.nelement() for p in parameters)) # num params in total
for p in parameters:
    p.requires_grad = True

76579


In [123]:
ix = torch.randint(0, Xtr.shape[0], (4,)) #look at 4 examples
Xb, Yb = Xtr[ix], Ytr[ix]
logits = model(Xb)
print(Xb.shape)
Xb

torch.Size([4, 8])


tensor([[ 0,  0,  0, 19,  1, 22,  9,  1],
        [ 0,  0, 20,  8,  1, 18, 21, 14],
        [ 0,  0,  0,  0,  0, 13,  1, 26],
        [ 0,  0,  0,  0,  0,  0,  0,  5]])

In [124]:
for layer in model.layers:
    print(layer.__class__.__name__, ':', tuple(layer.out.shape))

Embed : (4, 8, 24)
FlattenConsecutive : (4, 4, 48)
Linear : (4, 4, 128)
BatchNorm1d : (4, 4, 128)
Tanh : (4, 4, 128)
FlattenConsecutive : (4, 2, 256)
Linear : (4, 2, 128)
BatchNorm1d : (4, 2, 128)
Tanh : (4, 2, 128)
FlattenConsecutive : (4, 256)
Linear : (4, 128)
BatchNorm1d : (4, 128)
Tanh : (4, 128)
Linear : (4, 27)


In [125]:
model.layers[0].out.shape # out of embedding layer

torch.Size([4, 8, 24])

In [126]:
model.layers[1].out.shape # out of flatten layer

torch.Size([4, 4, 48])

In [127]:
model.layers[2].out.shape # out of linear layer

torch.Size([4, 4, 128])

In [129]:
#same optimization as last time
max_steps = 200000
batch_size = 32
lossi = []
ud = [] #update to data ratio

#put layers into train mode
for layer in model.layers:
    layer.training = True

for i in range(max_steps):
    #minibatch construct
    ix = torch.randint(0,Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]

    
    #forward pass
    loss = F.cross_entropy(model(Xb), Yb) #loss function

    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    #update
    lr = 0.1 if i < max_steps*0.75 else 0.01
    for p in parameters:
        p.data += -lr*p.grad

    if i % 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())
    with torch.no_grad():
        ud.append([(lr*p.grad.std() / p.data.std()).log10().item() for p in parameters])

      0/ 200000: 3.5391
  10000/ 200000: 2.0095
  20000/ 200000: 2.1224
  30000/ 200000: 2.2855
  40000/ 200000: 2.0020
  50000/ 200000: 2.0617
  60000/ 200000: 2.1278
  70000/ 200000: 2.1193
  80000/ 200000: 2.3880
  90000/ 200000: 2.0031
 100000/ 200000: 2.1111
 110000/ 200000: 2.2455
 120000/ 200000: 2.3378
 130000/ 200000: 1.8094
 140000/ 200000: 2.0651
 150000/ 200000: 2.1755
 160000/ 200000: 1.8919
 170000/ 200000: 1.7865
 180000/ 200000: 2.1288
 190000/ 200000: 1.9155


In [1]:
lossit = torch.tensor(lossi)
plt.plot(lossit.view(469, -1).mean(1))

NameError: name 'torch' is not defined

In [133]:
for layer in model.layers:
    print(layer.__class__.__name__, ':', tuple(layer.out.shape))

Embed : (1, 8, 24)
FlattenConsecutive : (1, 4, 48)
Linear : (1, 4, 128)
BatchNorm1d : (1, 4, 128)
Tanh : (1, 4, 128)
FlattenConsecutive : (1, 2, 256)
Linear : (1, 2, 128)
BatchNorm1d : (1, 2, 128)
Tanh : (1, 2, 128)
FlattenConsecutive : (1, 256)
Linear : (1, 128)
BatchNorm1d : (1, 128)
Tanh : (1, 128)
Linear : (1, 27)


In [134]:
#put layers into eval mode
for layer in model.layers:
    layer.training = False

@torch.no_grad() # decorator disables gradient tracking
def split_loss(split):
    x,y = {
        'train':(Xtr, Ytr),
        'val': (Xdev, Ydev),
        'test': (Xte, Yte),
    }[split]
    
 
    logits = model(x)
    loss = F.cross_entropy(logits, y) #softmax classification of logits, forward and backward much more efficient, fewer bugs
    print(split, loss.item())

split_loss('train')
split_loss('val')

train 1.9887913465499878
val 2.058871030807495


In [132]:
#sample from the model
for layer in model.layers:
    layer.training = False
    
for _ in range(20):

    out = []
    context = [0] * blocksize #init with all ...
    while True:
        #forward the NN
        logits = model(torch.tensor([context]))
        probs = F.softmax(logits, dim=1)
        #sample from distribution
        ix = torch.multinomial(probs, num_samples=1).item()
        #shift context window and track the samples
        context = context[1:] + [ix]
        out += [ix]
        #if we sample '.' break
        if ix == 0:
            break

    print(''.join(itos[i] for i in out))

pirston.
joarron.
tayel.
kartod.
elias.
deardie.
izaylah.
ebtavin.
denet.
mehyanre.
sinah.
marrem.
brillynt.
annello.
jaciey.
cuurie.
bloryston.
ailee.
tarros.
kamilo.
